In [1]:
!pip install 'pydantic[email]'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 8.4 MB/s eta 0:00:00


In [2]:
import sys
from enum import auto, IntFlag
from typing import Any
from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    SecretStr,
    ValidationError,
)

# Define permissões usando IntFlag
class Role(IntFlag):
    Author = auto()
    Editor = auto()
    Developer = auto()
    # Admin é a união bitwise de todos os papéis acima
    Admin = Author | Editor | Developer

# Modelo de dados principal que estrutura o Usuario
class User(BaseModel):
    name: str = Field(examples=["Arjan"])

    # Verifica o email e impede alterações após a criação
    email: EmailStr = Field(
        examples=["example@arjancodes.com"],
        description="The email address of the user",
        frozen=True,
    )

    # Mascarando o valor, protegendo a senha
    password: SecretStr = Field(
        examples=["Password123"], description="The password of the user"
    )

    # Define o papel do usuário
    role: Role = Field(default=None, description="The role of the user")

# Transformar um dicionário em um objeto User validado
def validate(data: dict[str, Any]) -> None:
    try:
        user = User.model_validate(data)
        print(user)
    except ValidationError as e:
        print("User is invalid")
        for error in e.errors():
            print(error)

def main() -> None:
    # Dados que seguem as regras do modelo (Sucesso)
    good_data = {
        "name": "Arjan",
        "email": "example@arjancodes.com",
        "password": "Password123",
    }

    # Dados incompletos e com e-mail inválido (Falha)
    bad_data = {"email": "", "password": ""}

    print("--- Validating Good Data ---")
    validate(good_data)

    print("\n--- Validating Bad Data ---")
    validate(bad_data)

if __name__ == "__main__":
    main()

--- Validating Good Data ---
name='Arjan' email='example@arjancodes.com' password=SecretStr('**********') role=None

--- Validating Bad Data ---
User is invalid
{'type': 'missing', 'loc': ('name',), 'msg': 'Field required', 'input': {'email': '', 'password': ''}, 'url': 'https://errors.pydantic.dev/2.12/v/missing'}
{'type': 'value_error', 'loc': ('email',), 'msg': 'value is not a valid email address: An email address must have an @-sign.', 'input': '', 'ctx': {'reason': 'An email address must have an @-sign.'}}


In [3]:
import enum
import hashlib
import re
from typing import Any
from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    field_validator,
    model_validator,
    SecretStr,
    ValidationError,
)

# Senha: Mínimo 8 caracteres, 1 maiúscula, 1 minúscula e 1 número
VALID_PASSWORD_REGEX = re.compile(r"^(?=.*[a-z])(?=.*[A-Z])(?=.*\d).{8,}$")
# Nome: Somente letras e pelo menos 2 caracteres
VALID_NAME_REGEX = re.compile(r"^[a-zA-Z]{2,}$")

class Role(enum.IntFlag):
    Author = 1
    Editor = 2
    Admin = 4
    SuperAdmin = 8

class User(BaseModel):
    name: str = Field(examples=["Arjan"])
    email: EmailStr = Field(
        examples=["user@arjancodes.com"],
        description="The email address of the user",
        frozen=True,
    )
    password: SecretStr = Field(
        examples=["Password123"], description="The password of the user"
    )
    role: Role = Field(
        default=None, description="The role of the user", examples=[1, 2, 4, 8]
    )

    # Verifica especificamente o campo 'name'
    @field_validator("name")
    @classmethod
    def validate_name(cls, v: str) -> str:
        if not VALID_NAME_REGEX.match(v):
            raise ValueError(
                "Name is invalid, must contain only letters and be at least 2 characters long"
            )
        return v

    # Converte String/Int para o Enum Role
    @field_validator("role", mode="before")
    @classmethod
    def validate_role(cls, v: int | str | Role) -> Role:
        # Mapeia o tipo de entrada para a forma correta de instanciar o Role
        op = {int: lambda x: Role(x), str: lambda x: Role[x], Role: lambda x: x}
        try:
            return op[type(v)](v)
        except (KeyError, ValueError):
            raise ValueError(
                f"Role is invalid, please use one of the following: {', '.join([x.name for x in Role])}"
            )

    # Valida o objeto inteiro
    @model_validator(mode="before")
    @classmethod
    def validate_user(cls, v: dict[str, Any]) -> dict[str, Any]:
        if "name" not in v or "password" not in v:
            raise ValueError("Name and password are required")

        if v["name"].casefold() in v["password"].casefold():
            raise ValueError("Password cannot contain name")

        if not VALID_PASSWORD_REGEX.match(v["password"]):
            raise ValueError(
                "Password is invalid, must contain 8 characters, 1 uppercase, 1 lowercase, 1 number"
            )

        # Importante: O Pydantic altera o valor original antes da criação do objeto
        v["password"] = hashlib.sha256(v["password"].encode()).hexdigest()
        return v

def validate(data: dict[str, Any]) -> None:
    try:
        user = User.model_validate(data)
        # O print mostrará a senha como '**********' (por ser SecretStr)
        # e o valor real será o hash SHA-256
        print(user)
    except ValidationError as e:
        print("User is invalid:")
        print(e)

def main() -> None:
    # Dicionário de testes com diversos cenários de erro e sucesso
    test_data = dict(
        good_data={
            "name": "Arjan",
            "email": "example@arjancodes.com",
            "password": "Password123",
            "role": "Admin",
        },
        bad_role={
            "name": "Arjan",
            "email": "example@arjancodes.com",
            "password": "Password123",
            "role": "Programmer",
        },
        bad_data={
            "name": "Arjan",
            "email": "bad email",
            "password": "bad password",
        },
        bad_name={
            "name": "Arjan<-_->",
            "email": "example@arjancodes.com",
            "password": "Password123",
        },
        duplicate={
            "name": "Arjan",
            "email": "example@arjancodes.com",
            "password": "Arjan123",
        },
        missing_data={
            "email": "",
            "password": "",
        },
    )

    for example_name, data in test_data.items():
        print(f"--- Testing: {example_name} ---")
        validate(data)
        print()

if __name__ == "__main__":
    main()

--- Testing: good_data ---
name='Arjan' email='example@arjancodes.com' password=SecretStr('**********') role=<Role.Admin: 4>

--- Testing: bad_role ---
User is invalid:
1 validation error for User
role
  Value error, Role is invalid, please use one of the following: Author, Editor, Admin, SuperAdmin [type=value_error, input_value='Programmer', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

--- Testing: bad_data ---
User is invalid:
1 validation error for User
  Value error, Password is invalid, must contain 8 characters, 1 uppercase, 1 lowercase, 1 number [type=value_error, input_value={'name': 'Arjan', 'email'...ssword': 'bad password'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

--- Testing: bad_name ---
User is invalid:
1 validation error for User
name
  Value error, Name is invalid, must contain only letters and be at least 2 characters long [type=value_error, input_value='Arj

In [4]:
import enum
import hashlib
import re
from typing import Any, Self
from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    field_serializer,
    field_validator,
    model_serializer,
    model_validator,
    SecretStr,
)

# Configurações de Regex para validação de formato
VALID_PASSWORD_REGEX = re.compile(r"^(?=.*[a-z])(?=.*[A-Z])(?=.*\d).{8,}$")
VALID_NAME_REGEX = re.compile(r"^[a-zA-Z]{2,}$")

class Role(enum.IntFlag):
    User = 0
    Author = 1
    Editor = 2
    Admin = 4
    SuperAdmin = 8

class User(BaseModel):
    name: str = Field(examples=["Example"])
    email: EmailStr = Field(
        examples=["user@arjancodes.com"],
        description="The email address of the user",
        frozen=True,
    )
    # exclude=True: Garante que a senha NUNCA apareça no dump/JSON final
    password: SecretStr = Field(
        examples=["Password123"], description="The password of the user", exclude=True
    )
    role: Role = Field(
        description="The role of the user",
        examples=[1, 2, 4, 8],
        default=0,
        validate_default=True, # Força a validação mesmo se usar o valor padrão
    )

    @field_validator("name")
    def validate_name(cls, v: str) -> str:
        if not VALID_NAME_REGEX.match(v):
            raise ValueError("Name must be letters only and at least 2 chars long")
        return v

    @field_validator("role", mode="before")
    @classmethod
    def validate_role(cls, v: int | str | Role) -> Role:
        # Permite converter entrada "Admin" (str) ou 4 (int) para o objeto Role
        op = {int: lambda x: Role(x), str: lambda x: Role[x], Role: lambda x: x}
        try:
            return op[type(v)](v)
        except (KeyError, ValueError):
            raise ValueError(f"Invalid Role. Options: {', '.join([x.name for x in Role])}")

    # Executado ANTES da criação do objeto (trabalha com o dicionário bruto)
    @model_validator(mode="before")
    @classmethod
    def validate_user_pre(cls, v: dict[str, Any]) -> dict[str, Any]:
        if "name" not in v or "password" not in v:
            raise ValueError("Name and password are required")
        if v["name"].casefold() in v["password"].casefold():
            raise ValueError("Password cannot contain name")
        if not VALID_PASSWORD_REGEX.match(v["password"]):
            raise ValueError("Password is too weak")

        # Transforma a senha em hash SHA-256 antes de instanciar o modelo
        v["password"] = hashlib.sha256(v["password"].encode()).hexdigest()
        return v

    # Executado DEPOIS que o objeto foi criado (trabalha com a instância 'self')
    @model_validator(mode="after")
    def validate_user_post(self) -> Self:
        # Regra de negócio específica: apenas um nome específico pode ser Admin
        if self.role == Role.Admin and self.name != "Arjan":
            raise ValueError("Only Arjan can be an admin")
        return self

    # SERIALIZADOR DE CAMPO: Define como o 'role' aparece no JSON
    @field_serializer("role", when_used="json")
    def serialize_role(self, v: Role) -> str:
        # Em vez de retornar o número (ex: 4), retorna o nome (ex: "Admin")
        return v.name

    # SERIALIZADOR DE MODELO: Controla a saída total do objeto quando convertido para JSON
    @model_serializer(mode="wrap", when_used="json")
    def serialize_user(self, serializer, info) -> dict[str, Any]:
        # Se não houver filtros (include/exclude), retorna uma versão simplificada
        if not info.include and not info.exclude:
            return {"name": self.name, "role": self.role.name}
        # Caso contrário, usa o serializador padrão do Pydantic
        return serializer(self)

def main() -> None:
    data = {
        "name": "Arjan",
        "email": "example@arjancodes.com",
        "password": "Password123",
        "role": "Admin",
    }

    # Criação do objeto usuário
    user = User.model_validate(data)

    if user:
        print("Dicionário Python (model_dump):", user.model_dump())
        print("Saída modo JSON:", user.model_dump(mode="json"))
        print("Saída JSON (sem cargo):", user.model_dump(exclude=["role"], mode="json"))
        print("Conversão dict(user):", dict(user))

if __name__ == "__main__":
    main()

Dicionário Python (model_dump): {'name': 'Arjan', 'email': 'example@arjancodes.com', 'role': <Role.Admin: 4>}
Saída modo JSON: {'name': 'Arjan', 'role': 'Admin'}
Saída JSON (sem cargo): {'name': 'Arjan', 'email': 'example@arjancodes.com'}
Conversão dict(user): {'name': 'Arjan', 'email': 'example@arjancodes.com', 'password': SecretStr('**********'), 'role': <Role.Admin: 4>}


In [5]:
from datetime import datetime
from typing import Optional
from uuid import uuid4

from fastapi import FastAPI
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient
from pydantic import BaseModel, EmailStr, Field, field_serializer, UUID4

# Inicializa a aplicação FastAPI
app = FastAPI()

class User(BaseModel):
    # Configuração para proibir campos extras que não estão definidos no modelo
    model_config = {
        "extra": "forbid",
    }

    # Lista estatica
    __users__ = []

    name: str = Field(..., description="Name of the user")
    email: EmailStr = Field(..., description="Email address of the user")

    # Listas de IDs, com limite máximo de 500 itens para performance/segurança
    friends: list[UUID4] = Field(
        default_factory=list, max_items=500, description="List of friends"
    )
    blocked: list[UUID4] = Field(
        default_factory=list, max_items=500, description="List of blocked users"
    )

    # Esses campos só podem ser passados via nome
    signup_ts: Optional[datetime] = Field(
        default_factory=datetime.now, description="Signup timestamp", kw_only=True
    )
    id: UUID4 = Field(
        default_factory=uuid4, description="Unique identifier", kw_only=True
    )

    # Serialização
    @field_serializer("id", when_used="json")
    def serialize_id(self, id: UUID4) -> str:
        return str(id)

# Retorna a lista de todos os usuários
@app.get("/users", response_model=list[User])
async def get_users() -> list[User]:
    return list(User.__users__)

# Cria um novo usuário e armazena na lista
@app.post("/users", response_model=User)
async def create_user(user: User):
    User.__users__.append(user)
    return user

# Busca um usuário específico ou retorna 404
@app.get("/users/{user_id}", response_model=User)
async def get_user(user_id: UUID4) -> User | JSONResponse:
    try:
        # Tenta encontrar o primeiro usuário com o ID correspondente
        return next((user for user in User.__users__ if user.id == user_id))
    except StopIteration:
        return JSONResponse(status_code=404, content={"message": "User not found"})

def main() -> None:
    # TestClient permite testar a API sem precisar subir um servidor real
    with TestClient(app) as client:
        # Criar 5 usuários em um loop
        for i in range(5):
            response = client.post(
                "/users",
                json={"name": f"User {i}", "email": f"example{i}@arjancodes.com"},
            )
            assert response.status_code == 200
            assert response.json()["name"] == f"User {i}"
            assert response.json()["id"]

        # Verificar se a listagem retorna os 5 usuários
        response = client.get("/users")
        assert response.status_code == 200
        assert len(response.json()) == 5

        # Criar um 6º usuário e buscá-lo individualmente pelo ID
        response = client.post(
            "/users", json={"name": "User 5", "email": "example5@arjancodes.com"}
        )
        user_id = response.json()['id']
        response = client.get(f"/users/{user_id}")
        assert response.status_code == 200
        assert response.json()["name"] == "User 5"

        # Buscar um ID que não existe (deve retornar 404)
        response = client.get(f"/users/{uuid4()}")
        assert response.status_code == 404
        assert response.json()["message"] == "User not found"

        # Tentar criar usuário com e-mail inválido (deve retornar 422 Unprocessable Entity)
        response = client.post("/users", json={"name": "User 6", "email": "wrong"})
        assert response.status_code == 422

    print("Todos os testes passaram com sucesso!")

if __name__ == "__main__":
    main()

Todos os testes passaram com sucesso!


/tmp/ipykernel_470/1941123841.py:26: PydanticDeprecatedSince20: `max_items` is deprecated and will be removed, use `max_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  friends: list[UUID4] = Field(
/tmp/ipykernel_470/1941123841.py:29: PydanticDeprecatedSince20: `max_items` is deprecated and will be removed, use `max_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  blocked: list[UUID4] = Field(
